# Feature–prediction correlation

This notebook asks which physical properties of a fixed-$k$ neighborhood correlate with held-out **window-level prediction correctness**. It does not use cell-level macro F1. All comparisons are stratified by `n_embeddings`; soma analyses are additionally separated by cell type. **This notebook never performs inference or feature extraction.** It reads compact analysis summaries from `analysis/feature_prediction_cache/`. The batch builder reuses the global per-window prediction cache at `/orcd/scratch/orcd/013/jcbliao/segclr/window_prediction_cache`; missing caches must be built through the Slurm batch script.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from analysis.feature_prediction_correlation import CACHE, completed_runs, load_summary

runs = completed_runs()
print(f'{len(runs)} completed runs:', *runs, sep='\n  ')

## Select runs and load precomputed caches

By default this selects one completed run at each embedding count, preferring mean when available. Replace `SELECTED_RUNS` to compare particular architectures. If a cache is missing, submit `sbatch scripts/sbatch/build_feature_prediction_cache.sh RUN_NAME`. Do not build it on the login node.

In [ ]:
def count_from_name(name):
    import re
    return int(re.search(r'_n(10|20|40)$', name).group(1))

SELECTED_RUNS = []
for n in (10, 20, 40):
    choices = [r for r in runs if count_from_name(r) == n]
    if choices:
        SELECTED_RUNS.append(next((r for r in choices if '_mean_' in r), choices[0]))

missing = [run for run in SELECTED_RUNS if not (CACHE / f'{run}.summary.json').exists()]
if missing:
    commands = [f'sbatch scripts/sbatch/build_feature_prediction_cache.sh {run}' for run in missing]
    raise FileNotFoundError('Missing compute-node caches. Submit these from the repo root:\n' + '\n'.join(commands))

summaries = [load_summary(run) for run in SELECTED_RUNS]
corr_df = pd.DataFrame([row for s in summaries for row in s['correlations']])
bins_df = pd.DataFrame([row for s in summaries for row in s['bins']])
cell_accuracy = pd.DataFrame([row for s in summaries for row in s['cells']])
print(f'Loaded compact summaries for {len(summaries)} runs; no raw windows loaded')
display(corr_df.head())

In [ ]:
def plot_binned(data, features, title, by_cell_type=False):
    part0 = data[data.cell_type.notna()] if by_cell_type else data[data.cell_type.isna()]
    groups = ['run', 'n_embeddings'] + (['cell_type'] if by_cell_type else [])
    for keys, part in part0.groupby(groups):
        fig, axes = plt.subplots(1, len(features), figsize=(5*len(features), 4))
        axes = np.atleast_1d(axes)
        for ax, feature in zip(axes, features):
            b = part[part.feature == feature].sort_values('bin')
            ax.plot(b.feature_median, b.accuracy, marker='o')
            ax.set(xlabel=feature.replace('_', ' '), ylabel='mean correctness')
            ax.grid(alpha=.25)
        fig.suptitle(title + ': ' + ' | '.join(map(str, np.atleast_1d(keys))))
        fig.tight_layout()
        plt.show()

## 1. Neighborhood extent, path length, and node density within each $k$

`geodesic_radius_um` is the farthest member's path distance from the center. `path_distance_um` is total skeleton cable in the induced neighborhood. Two density definitions are shown: nodes per µm of cable and nodes per spherical µm³ using the maximum Euclidean radius.

In [ ]:
extent_features = ['geodesic_radius_um', 'path_distance_um', 'node_density_per_um',
                   'spatial_radius_um', 'spatial_density_per_um3']
display(corr_df[corr_df.cell_type.isna() & corr_df.feature.isin(extent_features)])
plot_binned(bins_df, extent_features, 'Neighborhood geometry')

## 2. Distance of the center node from soma, separated by cell type

Spatial distance is center-to-nucleus Euclidean distance. Path distance uses the graph geodesic from the graph node nearest the nucleus to the center node. Disconnected components remain missing rather than being assigned a fabricated distance.

In [ ]:
soma_features = ['soma_path_um', 'soma_spatial_um']
soma_corr = corr_df[corr_df.cell_type.notna() & corr_df.feature.isin(soma_features)]
display(soma_corr.sort_values(['n_embeddings', 'cell_type', 'feature']))
plot_binned(bins_df, soma_features, 'Distance from soma', by_cell_type=True)

## 3. Segmentation volume contained in the embeddings

Per-node occupancy comes from `data/mask_volume_cache`, the saved count of cell-mask voxels inside the embedding input box. The plots compare sum, mean, and median across the $k$ members. The sum double-counts overlapping physical boxes and should be read as aggregate input occupancy, not union volume.

In [ ]:
volume_features = ['volume_sum_um3', 'volume_mean_um3', 'volume_median_um3']
display(corr_df[corr_df.cell_type.isna() & corr_df.feature.isin(volume_features)])
plot_binned(bins_df, volume_features, 'Embedding-box occupancy volume')

## 4. New-skeleton clearance radius

Each embedding node is joined by nearest coordinate to the **new** skeleton at `/orcd/scratch/orcd/013/jcbliao/skeletons/segclr/skeletons`; the skeleton's reported `radius` value is used. These are not CAVE skeleton radii. Mean, median, minimum, and maximum are calculated across each neighborhood.

In [ ]:
radius_features = ['radius_mean_nm', 'radius_median_nm', 'radius_min_nm', 'radius_max_nm']
display(corr_df[corr_df.cell_type.isna() & corr_df.feature.isin(radius_features)])
plot_binned(bins_df, radius_features, 'New-skeleton clearance radius')

## 5. Cell-ID effect in the validation set

Each point is one held-out cell's mean correctness across all of its center-node windows, sorted from lowest to highest. The companion distribution and quantiles show how much accuracy varies by specific cell ID.

In [ ]:
display(cell_accuracy.groupby(['run', 'n_embeddings']).accuracy.describe(
    percentiles=[.05, .1, .25, .5, .75, .9, .95]))
for (run, n), part in cell_accuracy.groupby(['run', 'n_embeddings']):
    part = part.sort_values('accuracy').reset_index(drop=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(np.arange(len(part)), part.accuracy, lw=1)
    axes[0].set(xlabel='held-out cells, sorted', ylabel='mean correctness', title='Per-cell ranked accuracy')
    axes[0].grid(alpha=.25)
    axes[1].hist(part.accuracy, bins=np.linspace(0, 1, 41))
    axes[1].set(xlabel='mean correctness', ylabel='cells', title='Distribution across cell IDs')
    fig.suptitle(f'{run} — n={n}')
    fig.tight_layout()
    plt.show()
display(cell_accuracy.sort_values(['run', 'accuracy']))